# Stackelberg-Duopol

We will analyze a two-firm market as the set up is given by the Stackelberg-Duopol (described in Stackelberg, H. (1934): Marktform und Gleichgewicht). We assume that both firms produce the same homogenous good. One of the firms produces first (is the Stackelberg leader). Then, the second firm produces after observing how much the leading firm produced. Since the leader firm knows that it is observed by the following firm, it incorporates this knowlegde into its decision process of how much of the good should be produced. By using subgame perfect Nash equilibrium we encounter how much of the good both firms produce in a Stackelberg-Duopol.

Imports and set magics:

In [19]:
import numpy as np
from scipy import optimize
import sympy as sm
import ipywidgets as widgets # for interactive plots/buttons

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

from darja_modelproject import stackelbergduopolClass
model = stackelbergduopolClass()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In general, we define L as the leader and F as the follower when having two firms: 
$$ L \in \{1,2\} \quad \text{and} \quad F \in \{1,2\} / \{L\}.$$ 


In the following, we assume that the first firm is the leader and the second firm is the follower. 

L chooses an amount of $x_{L}$, F chooses $x_{F}$ to produce. We assume that the two firms have different cost functions $C_{L} \neq  C_{F}$.

The Stackelberg Duopol is solved by backward induction. So, we first need to derive the best response function of the follower. 

F maximizes its protfit by: 
$$ \max_{x_{F} \geq 0} \quad P(x_{L}+ x_{F}) \cdot x_{F} - C_{F}(x_{F}) $$ 

To get the best response function of F we need to derive $\partial/\partial x_F $ and solve it such that $x_F$ can be written as a function of $x_L$.

$$ \partial/\partial x_F = 0 \Leftrightarrow x_{F}^*  =  ... $$

L anticipates the optimal solution of F, $x_{F}^*$, and maximizes its profit by: 

$$ \max_{x_{L} \geq 0} \quad P(x_{L} +x_{F}^*) \cdot x_{L} - C_{L}(x_L)$$ 

Finally, L gets the optimal solution of $x_{L}^*$ when solving the FOC of $ \partial/\partial x_L = 0 $.

**Write out the model in equations here.** 

Make sure you explain well the purpose of the model and comment so that other students who may not have seen it before can follow.  

## Analytical solution

There is an analytical solution for the Stackelberg Duopol. Therefore, we will first solve the model numerically by using sympy. 

We start by definining the parameters and variables in sympy: 

In [20]:
## solution using sympy 
x_1 = sm.symbols("x_1")
x_2 = sm.symbols("x_2")
a = sm.symbols("a")
#a = 5
b = sm.symbols("b")
#b = 1/4
p_1 = sm.symbols("p_1")
p_2 = sm.symbols("p_2")
#p_1 = 2
#p_2 = 1


Then we define the inverse demand function:

In [21]:
inverse_demand =  a-b*(x_1 + x_2)
inverse_demand

a - b*(x_1 + x_2)

Next, we define the objective function for the leading firm: 

In [22]:
objective_1 = inverse_demand * x_1 - p_1*x_1
objective_1

-p_1*x_1 + x_1*(a - b*(x_1 + x_2))

Now, we define the objective function for the following firm:

In [23]:
objective_2 = inverse_demand * x_2 - p_2*x_2
objective_2

-p_2*x_2 + x_2*(a - b*(x_1 + x_2))

The next step is to derive the objective function for the following firm regarding the amount x the firm consumes: 

In [24]:
foc = sm.diff(objective_2, x_2)
foc

a - b*x_2 - b*(x_1 + x_2) - p_2

Afterwards, we solve the derivative such that $x_2$ only depends on the amount that the leader firm consumes, $x_1$:

In [25]:
sol = sm.solve(sm.Eq(foc,0), x_2)
sol ## best answer function of firm 2 

[(a - b*x_1 - p_2)/(2*b)]

We substitute x_2 by the previous solution such that the objective function of the leading firm only depends on $x_1$:

In [26]:
sub_objective_1= objective_1.subs(x_2, sol[0])
sub_objective_1

-p_1*x_1 + x_1*(a - b*(x_1 + (a - b*x_1 - p_2)/(2*b)))

Finally, we can solve the FOC for the leading firm and get the optimal amount it produces: 

In [27]:
foc_1 = sm.diff(sub_objective_1, x_1)
foc_1
sol_1 = sm.solve(sm.Eq(foc_1,0), x_1)
sol_1

[(a - 2*p_1 + p_2)/(2*b)]

Now, we insert some values for the parameters a,b,c, $p_1$, $p_2$:
$$ a = 5 $$
$$ b = 1/4 $$ 
$$ p_1 = 2 $$
$$ p_2 = 1. $$

In [28]:
sol_1[0].subs(a, 5).subs(b, 1/4).subs(p_1,2).subs(p_2,1)

4.00000000000000

So the leading firm produces $x_L^*$ = 4 units of the good. 

Having $x_L^*$ = 4 we can directly find out how much the following firm will produce when it observes how much the leader produces: 

We just substitute the optimal amount of the leading firm in the best response function of the following firm: 

In [29]:
sol_2 = sol[0].subs(x_1, 4).subs(a, 5).subs(b, 1/4).subs(p_1,2).subs(p_2,1)
sol_2 

6.00000000000000

So this the amount both firms produce when the first firm is the leading firm: 
$$ x_L^*= 4 $$
$$ x_F^*= 6 $$

If your model allows for an analytical solution, you should provide here.

You may use Sympy for this. Then you can characterize the solution as a function of a parameter of the model.

To characterize the solution, first derive a steady state equation as a function of a parameter using Sympy.solve and then turn it into a python function by Sympy.lambdify. See the lecture notes for details. 

## Numerical solution

We still use the same price function and cost functions. 
This is a rather brute force approach but we get the same solution as trying analytically.

We use different techniques to solve the Stacklberg Duopol numerically. 
First, use a solver from scipy. Then we use a self-defined solver (similar to the one defined in the lecture). 
At the end we try to find the root of the first derivative of the leading firm's profit function which gives the optimal solution.

Here we use scipy:

In [30]:
def constraint_x(x):
    return x  # This ensures that x is non-negative


constraints ={'type': 'ineq', 'fun': constraint_x} ## adding constraint for x


## bounds for solutions
bounds = [(0,None)] ## non negative amount x

# c. call solver, use SLSQP
initial_guess = 6
## slsqp als method kann bounds und constrains annehmen
sol_case2 = optimize.minimize(
model.neg_objective_1, initial_guess, bounds = bounds ,constraints= constraints,
method='SLSQP')


In [31]:
sol_case2

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -1.9999999999999991
       x: [ 4.000e+00]
     nit: 3
     jac: [-5.960e-08]
    nfev: 6
    njev: 3

In [32]:
sol_case2.x

array([4.])

And the solution for the second firm as the follower:

In [33]:
model.best_func2(sol_case2.x) ## solution for firm 2

array([6.])

OR:

Here, we use a self-defined solver:

In [34]:
model.minimize_solver(20) ## own defined solver  

(4.0001073741824, 13, 79, 0)

It gives us the same solution as with using a solver from scipy. 

OR:

We only solve the FOC by finding the root of the first derivative of the leader's profit function.

In [35]:
## finding the root of the first derivative gives us the solution! 
optimize.root_scalar(model.derivative_1,x0=-5.0,method='newton')

      converged: True
           flag: converged
 function_calls: 3
     iterations: 1
           root: 4.0

You can always solve a model numerically. 

Define first the set of parameters you need. 

Then choose one of the optimization algorithms that we have gone through in the lectures based on what you think is most fitting for your model.

Are there any problems with convergence? Does the model converge for all starting values? Make a lot of testing to figure these things out. 

# Further analysis

In [39]:
widgets.interact(model.interactive_figure, 
                 p1 =widgets.FloatSlider(description=r"p1", min=0, max=5, step=0.05, value=2),
                 p2 = widgets.FloatSlider(description=r"p2", min=0, max=5, step=0.05, value=1), 
                 a_1 =widgets.FloatSlider(description=r"a", min=2, max=10, step=0.05, value=5), 
                 b_1=widgets.FloatSlider(description=r"b", min=0, max=2, step=0.05, value=1/4));

interactive(children=(FloatSlider(value=2.0, description='p1', max=5.0, step=0.05), FloatSlider(value=1.0, des…

Make detailed vizualizations of how your model changes with parameter values. 

Try to make an extension of the model. 

# Extension


We could look at 3 markets? Having a stackelberg oligopol? 

# Conclusion

Add concise conclusion. 